In [1]:
from pathlib import Path
import pandas as pd

# Cell 0: load all CSV files in a directory into a dict of DataFrames

def load_all_csv(path='.', recursive=False, show_progress=True, **read_csv_kwargs):
    """
    Load all .csv files from `path` into a dict of pandas.DataFrame objects.
    - path: directory path (str or Path)
    - recursive: if True, search subdirectories
    - show_progress: print loading status
    - read_csv_kwargs: passed to pandas.read_csv
    Returns: dict mapping filename (with extension) -> DataFrame
    """
    p = Path(path)
    pattern = '**/*.csv' if recursive else '*.csv'
    files = sorted(p.glob(pattern))
    dfs = {}
    for f in files:
        if show_progress:
            print(f"Reading {f} ...", end=' ')
        try:
            # Primary attempt: default pandas inference
            df = pd.read_csv(f, **read_csv_kwargs)
        except Exception:
            # Fallbacks: try to auto-detect delimiter and common encodings
            tried = False
            for enc in ('utf-8', 'latin1'):
                try:
                    df = pd.read_csv(f, sep=None, engine='python', encoding=enc, **read_csv_kwargs)
                    tried = True
                    break
                except Exception:
                    continue
            if not tried:
                if show_progress:
                    print("failed")
                print(f"Failed to read {f!s}; skipping.")
                continue
        dfs[f.name] = df
        if show_progress:
            print(f"ok (shape={df.shape})")
    if show_progress:
        print(f"Loaded {len(dfs)} CSV files from {p.resolve()}")
    return dfs

# Example usage:
# dfs = load_all_csv('.', recursive=True)
# Access a DataFrame by filename: dfs['my_file.csv']

In [2]:
# combine csv files into one dataframe with columns ['q_id',"source_lang", "target_lang", q_src,q_tgt,a_src,a_tgt,correct_target]
def combine_csv_files(dfs):
    combined_df = pd.DataFrame()
    for filename, df in dfs.items():
        if all(col in df.columns for col in ['q_id', 'source_lang', 'target_lang', 'q_src', 'q_tgt', 'a_src', 'a_tgt', 'correct_target']):
            combined_df = pd.concat([combined_df, df[['q_id', 'source_lang', 'target_lang', 'q_src', 'q_tgt', 'a_src', 'a_tgt', 'correct_target']]], ignore_index=True)
        else:
            print(f"Skipping {filename}: missing required columns.")
    return combined_df

all_eval_dfs = load_all_csv('../../eval/artifacts', recursive=True)
print(f"Total files loaded: {len(all_eval_dfs)}")

combined_eval_df = combine_csv_files(all_eval_dfs) 
print(f"Combined DataFrame shape: {combined_eval_df.shape}")
combined_eval_df.head()

Reading ../../eval/artifacts/all_metrics_summary.csv ... ok (shape=(12, 4))
Reading ../../eval/artifacts/de_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/en_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/en_source_answers.csv ... ok (shape=(39, 4))
Reading ../../eval/artifacts/es_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/fr_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/he_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/hi_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/id_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/it_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/ja_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/ko_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/pt_predictions.csv ... ok (shape=(39, 9))
Reading ../../eval/artifacts/zh_predictions.csv ... ok (shape=(39, 9)

,q_id,source_lang,target_lang,q_src,q_tgt,a_src,a_tgt,correct_target
0,33,en,de,In which country was AFN Bremerhaven located?,In welchem Land befand sich AFN Bremerhaven?,AFN Bremerhaven was located in Germany. AFN st...,AFN Bremerhaven befand sich in Deutschland. AF...,True
1,30,en,de,What was the position of August Joseph Donatel...,Welche Position hatte August Joseph Donatelli ...,August Joseph Donatelli served as a waist gunn...,August Joseph Donatelli war während des Zweite...,False
2,55,en,de,What was the television network that aired The...,Welcher Fernsehsender strahlte Die größte kana...,"The television network that aired ""The Greates...","Die Dokumentarserie ""Die größte kanadische Erf...",False
3,39,en,de,Since which decade did bowl games start counti...,Ab welchem Jahrzehnt wurden Bowl-Spiele in die...,Bowl games started counting toward single-seas...,Bowl-Spiele wurden ab den 2000er Jahren in die...,True
4,63,en,de,What is the original name of Entebbe General H...,Wie lautete der ursprüngliche Name des Allgeme...,The original name of Entebbe General Hospital ...,Der ursprüngliche Name des Allgemeinen Kranken...,False


In [3]:
# load all features datasets
features_dfs = load_all_csv('.', recursive=False)
print(f"Total feature files loaded: {len(features_dfs)}")
for filename, df in features_dfs.items():
    print(f"{filename}: shape={df.shape}")

Reading eclektic_long_article_language_counts.csv ... ok (shape=(468, 4))
Reading eclektic_long_qa_topics.csv ... ok (shape=(468, 6))
Reading eclektic_long_subset.csv ... ok (shape=(468, 12))
Reading eclektic_long_subset_with_question_type.csv ... ok (shape=(468, 13))
Reading eclektic_long_with_cooc_features.csv ... ok (shape=(468, 13))
Reading syntactic_complexity.csv ... ok (shape=(468, 18))
Loaded 6 CSV files from /Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/processed
Total feature files loaded: 6
eclektic_long_article_language_counts.csv: shape=(468, 4)
eclektic_long_qa_topics.csv: shape=(468, 6)
eclektic_long_subset.csv: shape=(468, 12)
eclektic_long_subset_with_question_type.csv: shape=(468, 13)
eclektic_long_with_cooc_features.csv: shape=(468, 13)
syntactic_complexity.csv: shape=(468, 18)


In [4]:
features_dfs["eclektic_long_article_language_counts.csv"].drop_duplicates(inplace=True)
temp_df = features_dfs["eclektic_long_article_language_counts.csv"][['q_id', 'language_version_count']]

merged_df = pd.merge(combined_eval_df, temp_df, on='q_id', how='left')
merged_df.shape




(468, 9)

In [5]:
features_dfs["eclektic_long_qa_topics.csv"].drop_duplicates(inplace=True)
temp_df = features_dfs["eclektic_long_qa_topics.csv"][['q_id', 'qa_topic']]
merged_df = pd.merge(merged_df, temp_df, on='q_id', how='left')

In [6]:
merged_df.shape


(468, 10)

In [7]:
temp_df = features_dfs["eclektic_long_subset_with_question_type.csv"][['q_id','question_type']]
temp_df.drop_duplicates(inplace=True)
merged_df = pd.merge(merged_df, temp_df, on='q_id', how='left')
merged_df.shape


/var/folders/ks/t4xykdjd3qj06ylqcn6zkprm0000gn/T/ipykernel_23485/1970492012.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df.drop_duplicates(inplace=True)


(468, 11)

In [9]:
merged_df.head()

,q_id,source_lang,target_lang,q_src,q_tgt,a_src,a_tgt,correct_target,language_version_count,qa_topic,question_type
0,33,en,de,In which country was AFN Bremerhaven located?,In welchem Land befand sich AFN Bremerhaven?,AFN Bremerhaven was located in Germany. AFN st...,AFN Bremerhaven befand sich in Deutschland. AF...,True,1,geography,location
1,30,en,de,What was the position of August Joseph Donatel...,Welche Position hatte August Joseph Donatelli ...,August Joseph Donatelli served as a waist gunn...,August Joseph Donatelli war während des Zweite...,False,1,history,event
2,55,en,de,What was the television network that aired The...,Welcher Fernsehsender strahlte Die größte kana...,"The television network that aired ""The Greates...","Die Dokumentarserie ""Die größte kanadische Erf...",False,1,culture,organization
3,39,en,de,Since which decade did bowl games start counti...,Ab welchem Jahrzehnt wurden Bowl-Spiele in die...,Bowl games started counting toward single-seas...,Bowl-Spiele wurden ab den 2000er Jahren in die...,True,1,sports,date_time
4,63,en,de,What is the original name of Entebbe General H...,Wie lautete der ursprüngliche Name des Allgeme...,The original name of Entebbe General Hospital ...,Der ursprüngliche Name des Allgemeinen Kranken...,False,2,health,definition


In [ ]:
temp_df = features_dfs["eclektic_long_with_translation_features.csv"]
merged_df = pd.merge(merged_df, features_dfs["eclektic_long_with_cooc_features.csv"], left_on=['q_id', 'target_lang'], right_on=['q_id', 'language'], how='left')
merged_df = merged_df.drop(columns=['question', 'answer', 'language'])
merged_df.head

In [20]:
features_dfs["syntactic_complexity.csv"]
merged_df = pd.merge(merged_df, features_dfs["syntactic_complexity.csv"], left_on=['q_id', 'target_lang'], right_on=['q_id', 'language'], how='left').drop(columns=['language', 'original_lang','original_content', 'original_question', 'original_answer', 'content', 'question', 'answer', 'translated', 'title', 'url'])

In [22]:
merged_df.columns
merged_df.shape

(468, 26)

In [23]:
merged_df.to_csv('./training_data_missing_macro.csv', index=False)